# 03 — EDA Main Storyline

Notebook này là bản EDA chính để đưa vào report/GitHub. Cấu trúc bám theo data storytelling:

1. Business pulse: doanh thu, margin, mùa vụ.
2. Product portfolio: danh mục/SKU nào kéo doanh thu và lợi nhuận.
3. Promotion effectiveness: promo tăng volume nhưng có đánh đổi margin hay không.
4. Returns and customer friction: lý do trả hàng và điểm đau vận hành.
5. Inventory action matrix: nhóm SKU nên reorder, hold, markdown, hoặc review.

Tất cả chart được lưu vào `reports/figures/`.

In [1]:
# ============================================================
# Shared setup
# ============================================================
from pathlib import Path
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# The notebooks are designed to run from repo/notebooks.
# They also work in Colab if you point PROJECT_ROOT or DATA_ROOT manually.
try:
    NOTEBOOK_DIR = Path.cwd().resolve()
except Exception:
    NOTEBOOK_DIR = Path(".").resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MARTS_DIR = PROJECT_ROOT / "data" / "marts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for d in [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, PROJECT_ROOT, Path("/mnt/data")]

def find_csv(filename: str) -> Path:
    """Find a CSV across standard repo folders and /mnt/data fallback."""
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find {filename}. Checked: {[str(d) for d in SEARCH_DIRS]}")

def read_csv(filename: str, parse_dates=None, **kwargs) -> pd.DataFrame:
    path = find_csv(filename)
    return pd.read_csv(path, parse_dates=parse_dates, low_memory=False, **kwargs)

def save_table(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=index)
    print(f"Saved table: {path}")
    return path

def save_fig(name: str, dpi: int = 160):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def money_fmt(x):
    if pd.isna(x):
        return "NA"
    if abs(x) >= 1e9:
        return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6:
        return f"{x/1e6:.2f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.2f}K"
    return f"{x:.2f}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("MARTS_DIR:", MARTS_DIR)

PROJECT_ROOT: C:\datathon
RAW_DIR: C:\datathon\data\raw
INTERIM_DIR: C:\datathon\data\interim
MARTS_DIR: C:\datathon\data\marts


## 1. Load analytical marts

Ưu tiên đọc mart đã tạo bởi notebook 02. Nếu chưa có, notebook sẽ fallback về file upload/interim cùng tên.

In [2]:
def read_any(filename, parse_dates=None):
    return read_csv(filename, parse_dates=parse_dates)

daily = read_any("5_daily_business_panel.csv", parse_dates=["Date"])
product_360 = read_any("4_product_360.csv")
product_month = read_any("6_product_month_panel.csv", parse_dates=["snapshot_date"])
try:
    item_fact = read_any("1_fact_order_item_enriched.csv", parse_dates=["order_date", "ship_date", "delivery_date", "first_return_date", "first_review_date"])
except Exception:
    # Fallback: build a lightweight item fact from raw files
    items = read_csv("order_items.csv")
    orders = read_csv("orders.csv", parse_dates=["order_date"])
    products = read_csv("products.csv")
    item_fact = items.merge(orders, on="order_id", how="left").merge(products, on="product_id", how="left")
    item_fact["line_revenue"] = item_fact["quantity"] * item_fact["unit_price"]
    item_fact["line_cogs"] = item_fact["quantity"] * item_fact["cogs"]
    item_fact["gross_profit"] = item_fact["line_revenue"] - item_fact["line_cogs"]
    item_fact["gross_margin"] = np.where(item_fact["line_revenue"] > 0, item_fact["gross_profit"] / item_fact["line_revenue"], np.nan)
    item_fact["has_promo"] = item_fact["promo_id"].notna() | item_fact["promo_id_2"].notna()

summary_metrics = {
    "date_min": daily["Date"].min().date(),
    "date_max": daily["Date"].max().date(),
    "total_revenue": daily["Revenue"].sum(),
    "total_cogs": daily["COGS"].sum(),
    "gross_profit": (daily["Revenue"] - daily["COGS"]).sum(),
    "gross_margin": (daily["Revenue"].sum() - daily["COGS"].sum()) / daily["Revenue"].sum(),
    "n_products": product_360["product_id"].nunique(),
    "n_item_lines": len(item_fact),
}
pd.DataFrame([summary_metrics])

ValueError: Missing column provided to 'parse_dates': 'Date'

## 2. Business pulse: revenue trend and margin

**Question:** doanh nghiệp tăng trưởng và biến động như thế nào qua thời gian?

Chart này dùng làm mở bài: doanh thu là mục tiêu forecast ở phần 3, còn margin cho biết doanh thu tăng có đi kèm chất lượng lợi nhuận hay không.

In [ ]:
monthly = (daily
    .set_index("Date")
    .resample("M")
    .agg(Revenue=("Revenue", "sum"), COGS=("COGS", "sum"))
    .reset_index()
)
monthly["gross_margin"] = (monthly["Revenue"] - monthly["COGS"]) / monthly["Revenue"]
monthly["revenue_3m_avg"] = monthly["Revenue"].rolling(3, min_periods=1).mean()
monthly["margin_3m_avg"] = monthly["gross_margin"].rolling(3, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(monthly["Date"], monthly["Revenue"], linewidth=1.2, alpha=0.45, label="Monthly revenue")
ax1.plot(monthly["Date"], monthly["revenue_3m_avg"], linewidth=2.5, label="3-month revenue average")
ax1.set_title("Business pulse: revenue trend with 3-month smoothing")
ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(monthly["Date"], monthly["margin_3m_avg"], linestyle="--", linewidth=2, label="3-month gross margin")
ax2.set_ylabel("Gross margin")
ax2.legend(loc="upper right")
save_fig("03_01_revenue_trend_margin.png")
plt.show()

monthly.tail()

**Interpretation template:**  
Doanh thu có xu hướng tăng dài hạn nhưng không đều; dùng rolling average giúp tách trend khỏi noise ngày/tháng. Margin nên được đọc song song để tránh tối ưu doanh thu bằng cách hy sinh lợi nhuận.

## 3. Seasonality: months that consistently outperform

**Question:** tháng nào nên được ưu tiên inventory/promo/logistics?

In [ ]:
season = (daily
    .assign(month=daily["Date"].dt.month)
    .groupby("month", as_index=False)
    .agg(avg_daily_revenue=("Revenue", "mean"), avg_margin=("gross_margin", "mean"), days=("Date", "count"))
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(season["month"], season["avg_daily_revenue"], alpha=0.75, label="Avg daily revenue")
ax1.set_title("Seasonality: average daily revenue and margin by month")
ax1.set_xlabel("Month")
ax1.set_ylabel("Average daily revenue")
ax1.set_xticks(range(1, 13))

ax2 = ax1.twinx()
ax2.plot(season["month"], season["avg_margin"], marker="o", linewidth=2, label="Avg gross margin")
ax2.set_ylabel("Average gross margin")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
save_fig("03_02_monthly_seasonality.png")
plt.show()

season.sort_values("avg_daily_revenue", ascending=False).head(5)

**Business implication:** các tháng có revenue cao nhưng margin thấp là vùng cần kiểm soát promo/discount; các tháng vừa revenue cao vừa margin ổn là thời điểm nên chuẩn bị tồn kho và logistics sớm.

## 4. Product portfolio: category contribution vs margin

**Question:** doanh nghiệp đang phụ thuộc vào nhóm sản phẩm nào, và nhóm đó có profitable không?

In [ ]:
cat = (item_fact
    .groupby("category", as_index=False)
    .agg(revenue=("line_revenue", "sum"), cogs=("line_cogs", "sum"), units=("quantity", "sum"), item_lines=("order_id", "count"))
)
cat["gross_profit"] = cat["revenue"] - cat["cogs"]
cat["gross_margin"] = cat["gross_profit"] / cat["revenue"]
cat["revenue_share"] = cat["revenue"] / cat["revenue"].sum()
cat = cat.sort_values("revenue", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cat["category"], cat["revenue"])
for i, row in cat.reset_index(drop=True).iterrows():
    ax.text(row["revenue"], i, f" {row['revenue_share']:.1%} | margin {row['gross_margin']:.1%}", va="center")
ax.set_title("Product portfolio: revenue share and gross margin by category")
ax.set_xlabel("Revenue")
ax.set_ylabel("Category")
save_fig("03_03_category_revenue_margin.png")
plt.show()

cat.sort_values("revenue", ascending=False)

**Business implication:** nếu một category chiếm tỷ trọng doanh thu quá lớn, mọi quyết định inventory/promo/return của category đó sẽ ảnh hưởng đáng kể đến toàn bộ P&L. Đây là lý do nên phân tích tiếp ở cấp SKU.

## 5. SKU Pareto: which products really matter?

**Question:** có bao nhiêu SKU tạo phần lớn doanh thu?

In [ ]:
prod = product_360.copy()
if "revenue" not in prod.columns:
    prod["revenue"] = prod.get("line_revenue", 0)
prod = prod.sort_values("revenue", ascending=False).reset_index(drop=True)
prod["rank"] = np.arange(1, len(prod) + 1)
prod["cum_revenue_share"] = prod["revenue"].cumsum() / prod["revenue"].sum()
prod["abc_class"] = pd.cut(prod["cum_revenue_share"], bins=[0, 0.80, 0.95, 1.01], labels=["A", "B", "C"], include_lowest=True)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(prod["rank"], prod["cum_revenue_share"], linewidth=2)
ax.axhline(0.80, linestyle="--", linewidth=1)
ax.axhline(0.95, linestyle="--", linewidth=1)
ax.set_title("SKU Pareto curve: cumulative revenue concentration")
ax.set_xlabel("SKU rank by revenue")
ax.set_ylabel("Cumulative revenue share")
ax.set_ylim(0, 1.02)
save_fig("03_04_sku_pareto.png")
plt.show()

abc_summary = prod.groupby("abc_class", observed=True).agg(
    n_skus=("product_id", "nunique"),
    revenue=("revenue", "sum"),
    avg_margin=("gross_margin", "mean") if "gross_margin" in prod.columns else ("revenue", "mean")
).reset_index()
abc_summary["revenue_share"] = abc_summary["revenue"] / abc_summary["revenue"].sum()
save_table(abc_summary, "03_abc_sku_summary.csv")
abc_summary

**Business implication:** nhóm A cần service level cao, tránh stockout; nhóm C cần kiểm soát overstock/markdown/discontinue để giảm vốn tồn kho bị khoá.

## 6. Promotion effectiveness: revenue lift vs margin pressure

**Question:** promo đang giúp tăng doanh thu hay chỉ làm mỏng margin?

In [ ]:
promo = (item_fact
    .assign(has_promo=item_fact["has_promo"].astype(bool))
    .groupby("has_promo", as_index=False)
    .agg(item_lines=("order_id", "count"), units=("quantity", "sum"), revenue=("line_revenue", "sum"), cogs=("line_cogs", "sum"), discount=("discount_amount", "sum"))
)
promo["gross_profit"] = promo["revenue"] - promo["cogs"]
promo["gross_margin"] = promo["gross_profit"] / promo["revenue"]
promo["revenue_share"] = promo["revenue"] / promo["revenue"].sum()
promo["line_share"] = promo["item_lines"] / promo["item_lines"].sum()
promo["label"] = promo["has_promo"].map({True: "Promo lines", False: "Non-promo lines"})

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(promo))
width = 0.35
ax.bar(x - width/2, promo["revenue_share"], width, label="Revenue share")
ax.bar(x + width/2, promo["gross_margin"], width, label="Gross margin")
ax.set_xticks(x)
ax.set_xticklabels(promo["label"])
ax.set_title("Promotion trade-off: revenue contribution vs gross margin")
ax.set_ylabel("Share / rate")
ax.legend()
save_fig("03_05_promo_revenue_margin_tradeoff.png")
plt.show()

promo

**Business implication:** promo nên được phân tầng: dùng để kích cầu ở nhóm cần tăng sell-through, nhưng hạn chế giảm sâu ở SKU A hoặc SKU có margin thấp.

## 7. Returns: what creates customer friction?

**Question:** nguyên nhân trả hàng nào lớn nhất và liên quan đến nhóm sản phẩm/kích cỡ nào?

In [ ]:
if "top_return_reason" in item_fact.columns:
    returns_view = item_fact[item_fact["top_return_reason"].notna()].copy()
    reason_col = "top_return_reason"
else:
    returns_raw = read_csv("returns.csv", parse_dates=["return_date"])
    products_raw = read_csv("products.csv")
    returns_view = returns_raw.merge(products_raw, on="product_id", how="left")
    reason_col = "return_reason"

reason = returns_view[reason_col].value_counts().reset_index()
reason.columns = ["return_reason", "records"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(reason["return_reason"][::-1], reason["records"][::-1])
ax.set_title("Return friction: top reasons for product returns")
ax.set_xlabel("Return records")
ax.set_ylabel("Return reason")
save_fig("03_06_return_reasons.png")
plt.show()

size_reason = (returns_view.groupby(["size", reason_col], as_index=False).size()
               .rename(columns={"size": "return_records", reason_col: "return_reason"}) if False else None)
reason

**Business implication:** nếu `wrong_size` là lý do chính, đây không chỉ là vấn đề hậu mãi mà là vấn đề conversion, mô tả sản phẩm và sizing UX. Action tốt hơn là giảm return từ đầu: size guide, fit recommendation, ảnh/model theo dáng người, cảnh báo size lệch chuẩn.

## 8. Inventory action matrix: combine demand and supply risk

**Question:** SKU nào nên reorder, SKU nào nên markdown/hold?

In [ ]:
pm = product_month.copy()
# robust revenue column
if "item_revenue" not in pm.columns:
    pm["item_revenue"] = 0

sku_ops = (pm.groupby(["product_id", "product_name", "category", "segment"], as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        stock_on_hand=("stock_on_hand", "mean"),
        stockout_days=("stockout_days", "mean"),
        stockout_rate=("stockout_flag", "mean"),
        overstock_rate=("overstock_flag", "mean"),
        reorder_rate=("reorder_flag", "mean"),
        sell_through_rate=("sell_through_rate", "mean"),
        revenue=("item_revenue", "sum")
    )
)

# If product_month revenue was not populated, bring product_360 revenue in.
if sku_ops["revenue"].sum() == 0 and "revenue" in product_360.columns:
    sku_ops = sku_ops.drop(columns=["revenue"]).merge(product_360[["product_id", "revenue"]], on="product_id", how="left")
    sku_ops["revenue"] = sku_ops["revenue"].fillna(0)

revenue_cut = sku_ops["revenue"].quantile(0.70)
stockout_cut = sku_ops["stockout_rate"].quantile(0.70)
overstock_cut = sku_ops["overstock_rate"].quantile(0.70)

conditions = [
    (sku_ops["revenue"] >= revenue_cut) & (sku_ops["stockout_rate"] >= stockout_cut),
    (sku_ops["revenue"] >= revenue_cut) & (sku_ops["stockout_rate"] < stockout_cut),
    (sku_ops["revenue"] < revenue_cut) & (sku_ops["overstock_rate"] >= overstock_cut),
]
choices = ["Reorder / protect", "Monitor / maintain", "Markdown / rationalize"]
sku_ops["action"] = np.select(conditions, choices, default="Review")

plot_df = sku_ops.copy()
if len(plot_df) > 1500:
    plot_df = pd.concat([
        plot_df.nlargest(500, "revenue"),
        plot_df.sample(1000, random_state=42)
    ]).drop_duplicates("product_id")

fig, ax = plt.subplots(figsize=(11, 6))
for action, g in plot_df.groupby("action"):
    sizes = np.sqrt(g["revenue"].clip(lower=0) + 1)
    if sizes.max() > 0:
        sizes = 20 + 180 * sizes / sizes.max()
    ax.scatter(g["sell_through_rate"], g["stockout_rate"], s=sizes, alpha=0.45, label=action)
ax.set_title("SKU action matrix: demand velocity vs stockout risk")
ax.set_xlabel("Average sell-through rate")
ax.set_ylabel("Stockout rate")
ax.legend(title="Suggested action", bbox_to_anchor=(1.02, 1), loc="upper left")
save_fig("03_07_sku_action_matrix.png")
plt.show()

action_summary = sku_ops.groupby("action", as_index=False).agg(
    n_skus=("product_id", "nunique"),
    revenue=("revenue", "sum"),
    avg_stockout_rate=("stockout_rate", "mean"),
    avg_overstock_rate=("overstock_rate", "mean"),
    avg_sell_through=("sell_through_rate", "mean")
).sort_values("revenue", ascending=False)
action_summary["revenue_share"] = action_summary["revenue"] / action_summary["revenue"].sum()
save_table(action_summary, "03_sku_action_summary.csv")
action_summary

## 9. Main EDA conclusion

**Storyline để đưa vào report:**

1. Doanh thu tăng nhưng có mùa vụ rõ, nên forecast/inventory cần calendar features thay vì chỉ dùng trend.
2. Revenue tập trung ở một số category/SKU; nhóm SKU A cần bảo vệ service level, nhóm long-tail cần tối ưu vốn tồn kho.
3. Promo đóng góp đáng kể vào doanh thu nhưng phải đọc cùng margin để tránh tăng doanh thu bằng discount quá sâu.
4. Return friction, đặc biệt nếu nghiêng về sizing, là điểm có thể hành động trực tiếp bằng UX/product content.
5. Inventory action matrix chuyển EDA từ mô tả sang prescriptive: reorder/protect, maintain, markdown/rationalize, review.